# Phase 1: AI Foundations & LLM Fundamentals
## Day 4: Decorators for Token Usage & Latency

### Core Theory (Just-in-Time)
Welcome to Day 4. In production AI Engineering, **observability** is non-negotiable. Two of the most critical metrics you must track for every LLM call are:
1. **Latency:** How long the LLM takes to respond. This directly impacts user experience (UX) and system throughput.
2. **Token Usage:** How many tokens (prompt + completion) were consumed. This directly impacts your unit economics and API rate limits.

To monitor these metrics cleanly without cluttering your core business logic, we use **Python Decorators**. A decorator is a structural design pattern (often known as a wrapper) that allows you to add cross-cutting concerns—like logging, timing, and token counting—to existing functions dynamically. 

When integrating with LangChain, tracking these metrics can be done easily via standard callbacks (like `get_openai_callback`). By packaging this tracking into a reusable decorator, we ensure every LLM invocation across our application is consistently monitored.

### Code Implementation
Below is a production-grade implementation of a decorator that captures both execution time and token usage via LangChain's callback system. Notice the use of `functools.wraps` to preserve the original function's metadata, strict type hints, and standard logging.

In [ ]:
import time
import logging
from functools import wraps
from typing import Callable, Any

# Using LangChain's standard callback for OpenAI token counting
from langchain_community.callbacks.manager import get_openai_callback
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Configure standard python logging (production best practice over print statements)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

def measure_llm_performance(func: Callable) -> Callable:
    """
    A decorator to measure the latency and token usage of a LangChain LLM call.
    It intercepts the wrapped function, records precise execution time, and 
    extracts token counts using LangChain's callback context.
    """
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()  # High-resolution performance counter
        
        # Use LangChain's callback manager to track OpenAI token usage
        with get_openai_callback() as cb:
            result = func(*args, **kwargs)
            
        end_time = time.perf_counter()
        latency = end_time - start_time
        
        # In production, these metrics would be sent to Datadog, Prometheus, etc.
        logger.info(
            f"[LLM Call: {func.__name__}] "
            f"Latency: {latency:.4f}s | "
            f"Tokens: {cb.total_tokens} (Prompt: {cb.prompt_tokens}, Completion: {cb.completion_tokens}) | "
            f"Cost: ${cb.total_cost:.6f}"
        )
        
        return result
        
    return wrapper

# Example Usage:
@measure_llm_performance
def generate_summary(text: str) -> str:
    """
    Generates a concise summary for the given text using an OpenAI model.
    """
    # Note: Requires OPENAI_API_KEY environment variable to be set
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    messages = [HumanMessage(content=f"Summarize this in one sentence: {text}")]
    
    response = llm.invoke(messages)
    return str(response.content)

### Practical Lab / Homework
Your actionable coding task for today:

1. **Async Adaptation**: The decorator above works for *synchronous* functions. In production, LLM applications are almost always heavily asynchronous to handle high concurrency. 
   - Write a new decorator named `@async_measure_llm_performance`.
   - Ensure it correctly wraps an `async def` function using `await`.
2. **Implement an Async Function**: Wrap an async LangChain method (like `llm.ainvoke()`) with your new decorator.
3. **Structured Logging**: Instead of a raw string, output the metrics as a structured JSON log. This is how modern telemetry stacks (like ELK or Datadog) prefer to ingest data.

*Write your code in the cell below to complete the lab.*

In [ ]:
# Write your Lab / Homework implementation here
import asyncio
import json
import time
import logging
from functools import wraps
from typing import Callable, Any

from langchain_community.callbacks.manager import get_openai_callback
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logger = logging.getLogger(__name__)

def async_measure_llm_performance(func: Callable) -> Callable:
    """
    An asynchronous decorator to measure the latency and token usage of a LangChain LLM call.
    Outputs metrics as structured JSON.
    """
    @wraps(func)
    async def wrapper(*args: Any, **kwargs: Any) -> Any:
        start_time = time.perf_counter()
        
        with get_openai_callback() as cb:
            result = await func(*args, **kwargs)
            
        end_time = time.perf_counter()
        latency = end_time - start_time
        
        log_data = {
            "event": "llm_call",
            "function_name": func.__name__,
            "latency_seconds": round(latency, 4),
            "total_tokens": cb.total_tokens,
            "prompt_tokens": cb.prompt_tokens,
            "completion_tokens": cb.completion_tokens,
            "cost_usd": round(cb.total_cost, 6)
        }
        logger.info(json.dumps(log_data))
        
        return result
        
    return wrapper

@async_measure_llm_performance
async def async_generate_summary(text: str) -> str:
    """
    Asynchronously generates a concise summary for the given text using an OpenAI model.
    """
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    messages = [HumanMessage(content=f"Summarize this in one sentence: {text}")]
    
    response = await llm.ainvoke(messages)
    return str(response.content)

# To run the async function, you would typically use:
# asyncio.run(async_generate_summary("Artificial Intelligence is rapidly transforming the engineering landscape."))

### Common Pitfalls
Here is what typically goes wrong in production when measuring these metrics:

1. **Sync Decorators on Async Functions:** Applying a standard wrapper to an `async` function returns a coroutine object immediately without waiting for execution. This logs zero execution time and counts zero tokens. You must use `async def wrapper(*args, **kwargs):` and `await func(...)` for async code.
2. **Streaming Tokens:** By default, OpenAI API responses that use `stream=True` often do not return token counts in standard chunks. You must explicitly request token usage by setting stream options (e.g., `stream_options={"include_usage": True}`) and parsing the final chunks carefully. The default `get_openai_callback()` might miss streaming tokens if not configured properly.
3. **Using `time.time()` instead of `time.perf_counter()`:** `time.time()` is subject to system clock updates (NTP synchronization) which can skew latency measurements. Always use `time.perf_counter()` for precision profiling.
4. **Losing Function Signatures:** Forgetting to use `@wraps(func)` means your function loses its `__name__` and `__doc__` attributes, replacing them with the wrapper's name. This heavily breaks routing libraries like FastAPI or Streamlit caching mechanisms.